In [2]:
import selfies as sf
import pandas as pd
import torch
from rdkit import Chem
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# MolGen

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("zjunlp/MolGen-large")
model = AutoModelForSeq2SeqLM.from_pretrained("zjunlp/MolGen-large")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
df = pd.read_csv("../data/raw/experts_merged.smi", sep=" ", header=None)

total_smiles = []

for i, row in df.iterrows():
    selfie = sf.encoder(row[0])
    
    sf_input = tokenizer(selfie, return_tensors="pt").to(device)
    # beam search
    molecules = model.generate(input_ids=sf_input["input_ids"],
                              attention_mask=sf_input["attention_mask"],
                              max_length=96,
                              min_length=32,
                              num_return_sequences=100,
                              num_beams=100)

    selfies = [tokenizer.decode(g.cpu(), skip_special_tokens=True, clean_up_tokenization_spaces=True).replace(" ","") for g in molecules]

    smiles = [sf.decoder(s) for s in selfies]
    total_smiles.extend(smiles)

In [ ]:
df = pd.DataFrame(total_smiles)
df.columns = ["smiles"]
df.to_csv("../data/sampling/molgen_generated.csv", sep=",", index=None)

# Expert SMILES -> SELFIES

In [3]:
df = pd.read_csv("../data/raw/experts_merged.smi", sep=" ", header=None)
df.columns = ["smiles"]
df["selfies"] = df["smiles"].apply(sf.encoder)
df["selfies"].to_csv("../data/raw/experts_merged.slf", sep=" ", index=None, header=None)

df["molecules"] = df["smiles"].apply(Chem.MolFromSmiles)

# print selfies as python list that can be inserted into yaml config file
df.head(2)

,smiles,selfies,molecules
0,N#Cc%10ccc(/C=C/C(/C=C/c1ccc(C#N)cc1)/C=C/c9cc...,[N][#C][C][=C][C][=C][Branch2][#Branch2][O][/C...,<rdkit.Chem.rdchem.Mol object at 0x7f100cace8f0>
1,CC(C)c%19ccc(n2c(c1ccc(C#N)cc1)cc%17c2cc(c%16c...,[C][C][Branch1][C][C][C][=C][C][=C][Branch2][#...,<rdkit.Chem.rdchem.Mol object at 0x7f100cace260>


In [3]:
df.shape

(27, 3)

In [4]:
import sys
sys.path.append("..")
from modules.core.features.filters.point_group_symmetry_filter import PointGroupSymmetryFilter
from modules.core.features.filters.flatness_filter import FlatnessFilter

filterek = FlatnessFilter()
# pgsf = PointGroupSymmetryFilter("../data/symmetries/symmetry_translation.csv")

filtered = filterek.apply(df["molecules"].tolist())
# filtered

Num atoms: 93, flatness: 0.0028622313516700913
Num atoms: 126, flatness: 0.002929097326406876
Num atoms: 132, flatness: 0.003244909660907702
Num atoms: 60, flatness: 0.003475021858557175
Num atoms: 48, flatness: 0.0037239941229396866
Num atoms: 60, flatness: 0.5500972325894123
Num atoms: 42, flatness: 0.7602601812335936
Num atoms: 42, flatness: 0.8549575956790095
Num atoms: 48, flatness: 0.8911508623000085
Num atoms: 18, flatness: 0.9091625980854268
Num atoms: 120, flatness: 0.9793939765777798
Num atoms: 72, flatness: 1.1334097414758024
Num atoms: 54, flatness: 1.2698936767096503
Num atoms: 30, flatness: 1.2698936767096503
Num atoms: 30, flatness: 1.3864371699463616
Num atoms: 48, flatness: 1.4427908148375133
Num atoms: 48, flatness: 1.6624406374979928
Num atoms: 30, flatness: 2.6981751562382525
Num atoms: 72, flatness: 3.824884474248288
Num atoms: 84, flatness: 4.248906677411933
Num atoms: 92, flatness: 4.261563665973329
Num atoms: 93, flatness: 4.5249779682379305
Num atoms: 91, flatn

In [4]:
generated = pd.read_csv("../data/sampling/reinvent_filtered_molecules.csv")
generated["molecules"] = generated["smiles"].apply(Chem.MolFromSmiles)
pgsf = PointGroupSymmetryFilter("../data/symmetries/symmetry_translation.csv")

filtered = pgsf.apply(generated["molecules"].tolist()[:100])
len(filtered)

Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs
Point group symmetry: Cs


100

# CHEMBL

In [ ]:
!wget https://ftp.ebi.ac.uk/pub/databases/chembl/ChEMBLdb/latest/chembl_35_sqlite.tar.gz
!tar -xvzf chembl_35_sqlite.tar.gz

https://ftp.ebi.ac.uk/pub/databases/chembl/ChEMBLdb/latest/chembl_35_schema.png

In [2]:
# %%timeit
import sqlite3
import pandas as pd
# Create your connection.
cnx = sqlite3.connect('../data/chembl_35_sqlite/chembl_35.db')

df = pd.read_sql_query("SELECT * FROM main.compound_structures", cnx)

df.shape

(2474590, 5)

In [2]:
df.head()

,molregno,molfile,standard_inchi,standard_inchi_key,canonical_smiles
0,1,\n RDKit 2D\n\n 24 26 0 0 0 0...,InChI=1S/C17H12ClN3O3/c1-10-8-11(21-17(24)20-1...,OWRSAHYFSSNENM-UHFFFAOYSA-N,Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccccc1Cl
1,2,\n RDKit 2D\n\n 25 27 0 0 0 0...,InChI=1S/C18H12N4O3/c1-11-8-14(22-18(25)21-16(...,ZJYUMURGSZQFMH-UHFFFAOYSA-N,Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(C#N)cc1
2,3,\n RDKit 2D\n\n 25 27 0 0 0 0...,InChI=1S/C18H16ClN3O3/c1-10-7-14(22-18(25)21-1...,YOMWDCALSDWFSV-UHFFFAOYSA-N,Cc1cc(-n2ncc(=O)[nH]c2=O)cc(C)c1C(O)c1ccc(Cl)cc1
3,4,\n RDKit 2D\n\n 23 25 0 0 0 0...,InChI=1S/C17H13N3O3/c1-11-2-4-12(5-3-11)16(22)...,PSOPUAQFGCRDIP-UHFFFAOYSA-N,Cc1ccc(C(=O)c2ccc(-n3ncc(=O)[nH]c3=O)cc2)cc1
4,5,\n RDKit 2D\n\n 24 26 0 0 0 0...,InChI=1S/C17H12ClN3O3/c1-10-8-13(21-17(24)20-1...,KEZNSCMBVRNOHO-UHFFFAOYSA-N,Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(Cl)cc1


In [3]:
import pandas as pd
import selfies as sf
import multiprocessing as mp

# Function to encode SMILES to SELFIES, handling errors
def try_encoder(smiles):
    try:
        if smiles is not None:
            return sf.encoder(smiles)
        return None
    except sf.EncoderError:
        return None
    except Exception:
        return None

# Function to process a chunk of the Series
def process_chunk(chunk):
    return chunk.apply(try_encoder)

if 'canonical_smiles' in df.columns:
    num_processes = 12  # Adjust as needed

    # Split the Series into chunks
    chunk_size = (len(df) + num_processes - 1) // num_processes
    chunks = [df['canonical_smiles'].iloc[i * chunk_size:(i + 1) * chunk_size] for i in range(num_processes)]

    with mp.Pool(processes=num_processes) as pool:
        processed_chunks = pool.map(process_chunk, chunks)

    # Concatenate the processed chunks back into a single Series
    df["selfies"] = pd.concat(processed_chunks)

    print(df.isna().sum())

    # Drop rows with NaN
    df.dropna(inplace=True)
    print(df.shape)

else:
    print("The 'canonical_smiles' column is not found")

molregno               0
molfile                0
standard_inchi         9
standard_inchi_key     0
canonical_smiles       0
selfies               47
dtype: int64
(2474534, 6)


In [5]:
df["canonical_smiles"].to_csv("../data/raw/chembl_35_smiles.txt", sep=" ", index=None, header=None)
df["selfies"].to_csv("../data/raw/chembl_35_selfies.txt", sep=" ", index=None, header=None)

In [ ]:
fdf = pd.read_sql_query("SELECT * FROM main.compound_properties", cnx) #  tabela z cechami do pretrenowania ew. modelu.
fdf.columns

Index(['molregno', 'mw_freebase', 'alogp', 'hba', 'hbd', 'psa', 'rtb',
       'ro3_pass', 'num_ro5_violations', 'cx_most_apka', 'cx_most_bpka',
       'cx_logp', 'cx_logd', 'molecular_species', 'full_mwt', 'aromatic_rings',
       'heavy_atoms', 'qed_weighted', 'mw_monoisotopic', 'full_molformula',
       'hba_lipinski', 'hbd_lipinski', 'num_lipinski_ro5_violations',
       'np_likeness_score'],
      dtype='object')

In [9]:
# join df and fdf on compound_id
merged_df = pd.merge(df, fdf, on="molregno", how="inner")
merged_df.to_parquet("../data/chembl_35_sqlite/chembl_35.parquet")

In [11]:
pd.read_parquet("../data/chembl_35_sqlite/chembl_35.parquet").head()

,molregno,molfile,standard_inchi,standard_inchi_key,canonical_smiles,selfies,mw_freebase,alogp,hba,hbd,...,full_mwt,aromatic_rings,heavy_atoms,qed_weighted,mw_monoisotopic,full_molformula,hba_lipinski,hbd_lipinski,num_lipinski_ro5_violations,np_likeness_score
0,1,\n RDKit 2D\n\n 24 26 0 0 0 0...,InChI=1S/C17H12ClN3O3/c1-10-8-11(21-17(24)20-1...,OWRSAHYFSSNENM-UHFFFAOYSA-N,Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccccc1Cl,[C][C][=C][C][Branch1][=N][N][N][=C][C][=Branc...,341.75,2.11,5.0,1.0,...,341.75,3.0,24.0,0.74,341.0567,C17H12ClN3O3,6.0,1.0,0.0,-1.56
1,2,\n RDKit 2D\n\n 25 27 0 0 0 0...,InChI=1S/C18H12N4O3/c1-11-8-14(22-18(25)21-16(...,ZJYUMURGSZQFMH-UHFFFAOYSA-N,Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(C#N)cc1,[C][C][=C][C][Branch1][=N][N][N][=C][C][=Branc...,332.32,1.33,6.0,1.0,...,332.32,3.0,25.0,0.73,332.0909,C18H12N4O3,7.0,1.0,0.0,-1.59
2,3,\n RDKit 2D\n\n 25 27 0 0 0 0...,InChI=1S/C18H16ClN3O3/c1-10-7-14(22-18(25)21-1...,YOMWDCALSDWFSV-UHFFFAOYSA-N,Cc1cc(-n2ncc(=O)[nH]c2=O)cc(C)c1C(O)c1ccc(Cl)cc1,[C][C][=C][C][Branch1][=N][N][N][=C][C][=Branc...,357.80,2.27,5.0,2.0,...,357.80,3.0,25.0,0.75,357.0880,C18H16ClN3O3,6.0,2.0,0.0,-0.82
3,4,\n RDKit 2D\n\n 23 25 0 0 0 0...,InChI=1S/C17H13N3O3/c1-11-2-4-12(5-3-11)16(22)...,PSOPUAQFGCRDIP-UHFFFAOYSA-N,Cc1ccc(C(=O)c2ccc(-n3ncc(=O)[nH]c3=O)cc2)cc1,[C][C][=C][C][=C][Branch2][Ring1][O][C][=Branc...,307.31,1.46,5.0,1.0,...,307.31,3.0,23.0,0.74,307.0957,C17H13N3O3,6.0,1.0,0.0,-1.10
4,5,\n RDKit 2D\n\n 24 26 0 0 0 0...,InChI=1S/C17H12ClN3O3/c1-10-8-13(21-17(24)20-1...,KEZNSCMBVRNOHO-UHFFFAOYSA-N,Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(Cl)cc1,[C][C][=C][C][Branch1][=N][N][N][=C][C][=Branc...,341.75,2.11,5.0,1.0,...,341.75,3.0,24.0,0.74,341.0567,C17H12ClN3O3,6.0,1.0,0.0,-1.49


# GraphAF - torch drug

In [ ]:
# TODO: environment doesn't work so far with UV